## Config

### Install

In [0]:
%run ./00_utility

In [0]:
%run ../FASE1/00_utils

In [0]:
%pip install pandarallel --quiet
%pip install xgboost==3.2.0 --quiet

In [0]:
import ast
import json
import warnings

import mlflow
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pandarallel import pandarallel
from pyspark.sql.types import DoubleType, FloatType
from pyspark.sql.window import Window

pandarallel.initialize(nb_workers=16, progress_bar=False)

### Parameters

In [0]:
catalog = get_catalog()
print("Catalog: ", catalog)
# MODEL URI
MODEL_URI = f"models:/{catalog}.whatif.whatif_lev1_model/2"

# TABLES
RAI_TABLE = f"{catalog}.whatif.output_palinsesto_rai" # tabella con lo scraping del palinsesto rai
COMP_TABLE = f"{catalog}.whatif.output_palinsesto_competitor" # tabella con lo scraping dei canali competitors
HIST_TABLE = f"{catalog}.whatif.storico_programmi" # tabella storico auditel
OUTPUT_TABLE = f"{catalog}.whatif.out_palinsesto_predict_all_slots" # tabella per la predizione di tutti gli slots
OUTPUT_TABLE_PRIMETIME = f"{catalog}.whatif.out_palinsesto_predict" # tabella per la predizione degli slots di primetime

# PARAMETERS
CHANNELS = ['Rai 1', 'Rai 2', 'Rai 3']
START_PRIMETIME = 20 * 3600 + 30 *60
DAY_TO_PREDICT_FROM_NOW = 6 # oggi + DAY_TO_PREDICT_FROM_NOW è il giorno per cui viene fatta la predizione
CUTOFF_DAYS = 7 # finestra temporale da escludere nel calcolo delle features perchè troppo vicina al giorno da predirre
CUTOFF_NUM_OCCURRENCES = 10 # numero di occorrenze massimo utilizzato per il calcolo delle features
TIME_TOLERANCE = 3 # tolleranza in ore per fare il match tra programma passato e futuro
MIN_COMPETITOR_OVERLAP = 0.60 # overlap in percentuale per due programmi per essere considerati competitors
HISTORICAL_PROGRAMMING_CUTOFF_DATE = "2025-05-27" # data da cui partire per estrarre lo storico auditel

## Loading Palinsesto Storico e Futuro

In [0]:
# Carichiamo lo storico auditel 
df_programmi = read_df_programmi(
    table_name=HIST_TABLE,
    date_from=HISTORICAL_PROGRAMMING_CUTOFF_DATE
)

# Riformattiamo la colonna data per una migliore lettura
df_programmi['Data'] = df_programmi['Data'].dt.date

# Convertiamo in pyspark e aggiungiamo la colonna TipoGiorno
df_programmi = (
    spark.createDataFrame(df_programmi)
    .withColumn(
        "TipoGiorno",
        F.when(F.col("GiornoSettimana").isin(0, 1, 2, 3, 4), F.lit("Weekday"))
         .otherwise(F.lit("Weekend"))
    )
)

In [0]:
# Carichiamo e normalizziamo il palinsesto futuro di RAI
rai_future = spark.table(RAI_TABLE)

# Rinominiamo le colonne per uniformità
rai_future = (
    rai_future
    .withColumnRenamed('ora', 'Ora')
    .withColumnRenamed('genere_predominante', 'DES_GENERE_ESTESA_INT')
    .withColumnRenamed('giorno_settimana', 'GiornoSettimana')
    # Aggiungiamo la colonna TipoGiorno (Weekday/Weekend)
    .withColumn(
        'TipoGiorno',
        F.when(F.col("GiornoSettimana").isin(0, 1, 2, 3, 4), F.lit('Weekday')).otherwise(F.lit('Weekend'))
    )
    # Calcoliamo ORA_INIZIO_TRX in secondi
    .withColumn(
        "ORA_INIZIO_TRX",
        F.split("orario_inizio", ":").getItem(0).cast("int") * 3600 +
        F.split("orario_inizio", ":").getItem(1).cast("int") * 60
    )
    # Calcoliamo ORA_FINE_TRX in secondi
    .withColumn(
        "ORA_FINE_TRX",
        F.split("orario_fine", ":").getItem(0).cast("int") * 3600 +
        F.split("orario_fine", ":").getItem(1).cast("int") * 60
    )
)
# Rimuoviamo colonne non necessarie
rai_future = rai_future.drop('fascia_oraria', 'share_storico')

In [0]:
# Carichiamo e normalizziamo il palinsesto futuro dei canali competitors
comp_future = spark.table(COMP_TABLE)

# Rinominiamo le colonne per uniformità
comp_future = (
    comp_future
    .withColumnRenamed('ora', 'Ora')
    .withColumnRenamed('genere_predominante', 'DES_GENERE_ESTESA_INT')
    .withColumnRenamed('giorno_settimana', 'GiornoSettimana')
    # Aggiungiamo la colonna TipoGiorno (Weekday/Weekend)
    .withColumn(
        'TipoGiorno',
        F.when(F.col("GiornoSettimana").isin(0, 1, 2, 3, 4), F.lit('Weekday')).otherwise(F.lit('Weekend'))
    )
    # Calcoliamo ORA_INIZIO_TRX in secondi
    .withColumn(
        "ORA_INIZIO_TRX",
        F.split("orario_inizio", ":").getItem(0).cast("int") * 3600 +
        F.split("orario_inizio", ":").getItem(1).cast("int") * 60
    )
    # Calcoliamo ORA_FINE_TRX in secondi
    .withColumn(
        "ORA_FINE_TRX",
        F.split("orario_fine", ":").getItem(0).cast("int") * 3600 +
        F.split("orario_fine", ":").getItem(1).cast("int") * 60
    )
)
# Rimuoviamo colonne non necessarie
comp_future = comp_future.drop('fascia_oraria', 'share_storico')

In [0]:
# Uniamo storico auditel, palinsesto futuro RAI e palinsesto competitors
df_all = (
    df_programmi
    .unionByName(rai_future, allowMissingColumns=True)
    .unionByName(comp_future, allowMissingColumns=True)
)

In [0]:
# Convertiamo da pyspark a pandas
df_all = df_all.toPandas()

## Creazione Features

In [0]:
# Calcolo feature storiche di share per ciascun programma
print('Computing precise historical stats...')
df_all = (
    df_all
    .groupby(['programma_norm', 'Canale', 'TipoGiorno'])
    .parallel_apply(
        compute_stats_last_occurrences,
        suffix='Precise',
        cutoff_days=CUTOFF_DAYS,
        cutoff_num_occurrences=CUTOFF_NUM_OCCURRENCES,
        time_tolerance=TIME_TOLERANCE
    )
    .reset_index(drop=True)
)

In [0]:
# Calcolo feature storiche di share per i programmi precedenti/seguenti
print('Computing previous and next program stats...')
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    df_all = (
        df_all
        .groupby(['Canale'])
        .parallel_apply(
            compute_stats_prev_next_program, 
            suffix='Precise'
        )
        .reset_index(drop=True)
    )

df_programmi_pre = spark.createDataFrame(df_all)
print('Done!')

In [0]:
# Calcolo feature storiche di share per i programmi competitors
df_programmi_pre = add_competitor(
    df_programmi_pre,
    suffix='Precise',
    min_competitor_overlap=MIN_COMPETITOR_OVERLAP
)

for c in df_programmi_pre.columns:
    if c.endswith('Precise'):
        df_programmi_pre = df_programmi_pre.withColumnRenamed(c, c.replace('Precise', ''))

In [0]:
# Filtriamo in un dataframe i programmi per i quali non è possibile fare una previsione:
# Assenza di dati storici di share o share negativo
df_programmi_no_pred = df_programmi_pre.where(F.col('StoricoShare').isNull() | (F.col('StoricoShare') < 0))
# Share zero o positivo
df_programmi_pre = df_programmi_pre.where(F.col('StoricoSharePrecise') >= 0.0)

In [0]:
# Sostituiamo df_programmi con il dataset pre-processato
df_programmi_pred = df_programmi_pre

# Aggiungiamo ulteriori feature calcolate
df_programmi_pred = df_programmi_pred.withColumn(
    'HighValueShare', F.col('StoricoShare') >= 0.12
)
df_programmi_pred = df_programmi_pred.withColumn(
    'Programma_durata', F.col("ORA_FINE_TRX") - F.col("ORA_INIZIO_TRX")
)

# Convertiamo i valori assoluti di share in delta
for c in [
    'StoricoShareMax',
    'StoricoShareMin',
    'StoricoShareLast',
    'StoricoShare_prev',
    'StoricoShare_next',
    'StoricoShareLast_prev',
    'StoricoShareLast_next'
]:
    df_programmi_pred = df_programmi_pred.withColumn(
        c, F.col(c) - F.col('StoricoShare')
    )

# Convertiamo le feature temporali come categoriche
df_programmi_pred = (
    df_programmi_pred
    .withColumn('Ora', F.col('Ora').cast('string'))
    .withColumn('GiornoSettimana', F.col('GiornoSettimana').cast('string'))
    .withColumn('Mese', F.col('Mese').cast('string'))
)

# Creazione delle feature delta per la differenza del contributo ponderato di audience tra programma e competitor.
df_programmi_pred = (
    df_programmi_pred
    .withColumn(
        'StoricoShareUomini_delta',
        F.col('StoricoShareUomini') * F.col('StoricoShare') -
        F.col('StoricoShareUomini_competitor') * F.col('StoricoShare_competitor')
    )
    .withColumn(
        'StoricoShare_08_14_delta',
        F.col('StoricoShare_08_14') * F.col('StoricoShare') -
        F.col('StoricoShare_08_14_competitor') * F.col('StoricoShare_competitor')
    )
    .withColumn(
        'StoricoShare_15_24_delta',
        F.col('StoricoShare_15_24') * F.col('StoricoShare') -
        F.col('StoricoShare_15_24_competitor') * F.col('StoricoShare_competitor')
    )
    .withColumn(
        'StoricoShare_25_64_delta',
        F.col('StoricoShare_25_64') * F.col('StoricoShare') -
        F.col('StoricoShare_25_64_competitor') * F.col('StoricoShare_competitor')
    )
    .withColumn(
        'StoricoShare_65_plus_delta',
        F.col('StoricoShare_65_plus') * F.col('StoricoShare') -
        F.col('StoricoShare_65_plus_competitor') * F.col('StoricoShare_competitor')
    )
)

In [0]:
# Applichiamo un filtro sui canali e sui giorni da predirre
df_programmi_pred = (
    df_programmi_pred
    .where(F.col('Canale').isin(CHANNELS))
    .where(F.col('Data') >= F.current_date())
)

if df_programmi_pred.isEmpty():
    dbutils.notebook.exit('Nessun programma da predirre')

df_programmi_no_pred = (
    df_programmi_no_pred
    .where(F.col('Canale').isin(CHANNELS))
    .where(F.col('Data') >= F.current_date())
)

## Load Model

In [0]:
model = mlflow.pyfunc.load_model(MODEL_URI)

## Prepare inference dataset

In [0]:
# Carica la lista delle variabili categoriche richieste
with open('/Workspace/Shared/AVANADE_WhatIf/FASE1/categorical_features_ohe.json', 'r') as f:
    categorical_features_ohe = json.load(f)

# Applichiamo OHE
for col, mapping in categorical_features_ohe.items():
    print(f'OHE inference: {col}')
    for original_value, feature_name in mapping.items():
        df_programmi_pred = df_programmi_pred.withColumn(
            feature_name,
            F.when(F.col(col).cast('string') == F.lit(original_value), 1).otherwise(0)
        )

# Controlli sugli input 
signature = model.metadata.signature
expected_columns = [col.name for col in signature.inputs.inputs]
print(f'Expected cols: {len(expected_columns)}')

# Conversione a Pandas
inference_pd = df_programmi_pred.toPandas()

# Allinea il DataFrame di inference alle colonne attese. se mancanti --> aggiunte e valorizzate a 0; se extra ---> rimosse
X_inf = inference_pd.reindex(
    columns=expected_columns,
    fill_value=0
)

# Debug: print delle colonne extra e mancanti
extra_cols = set(inference_pd.columns) - set(expected_columns)
missing_cols = set(expected_columns) - set(inference_pd.columns)

print(f'Extra cols ignored: {len(extra_cols)}')
print(f'Missing cols added: {len(missing_cols)}')

# Maps spark types to pandas types
type_mapping = {
    "integer": "int32",
    "long": "int64",
    "float": "float32",
    "double": "float64",
    "boolean": "bool"
}
# conversione di ciascuna colonna nel tipo atteso dal modello
signature_types = {col.name: col.type.name for col in signature.inputs.inputs}

for col_name, mlflow_type in signature_types.items():
    pandas_type = type_mapping.get(mlflow_type, "float64")
    X_inf[col_name] = pd.to_numeric(X_inf[col_name], errors='coerce')
    if "int" in pandas_type:
        X_inf[col_name] = X_inf[col_name].fillna(0).astype(pandas_type)
    else:
        X_inf[col_name] = X_inf[col_name].astype(pandas_type)

# Cleanup finale
X_inf = X_inf.fillna(0)

# Predict

In [0]:
# Predizione share residuo
pred_residuo = model.predict(X_inf)
inference_pd['share_residuo'] = pred_residuo

# Calcolo share predetto
inference_pd['share_predetto'] = inference_pd['StoricoShare'] + inference_pd['share_residuo']

# Output

## Tutte le predizioni

In [0]:
# Rimuoviamo dalla tabella di inferenza e dalla tabelle dei programmi non predetti le colonne che non ci interessano
ohe_var_list = [v for mapping in categorical_features_ohe.values() for v in mapping.values()]
vosdal_var_list = inference_pd.columns[inference_pd.columns.str.contains("LiveVOSDAL")].tolist()

inference_pd = inference_pd.drop(vosdal_var_list, axis=1)
inference_pd = inference_pd.drop(ohe_var_list, axis=1)

df_programmi_no_pred = df_programmi_no_pred.drop(*vosdal_var_list)

In [0]:
# Conversione da pandas a spark
inference_spark = spark.createDataFrame(inference_pd)

# Cast share_residuo and share_predetto columns nel dataframe dei programmi senza predizioni
df_programmi_no_pred = df_programmi_no_pred.withColumn("share_residuo",F.lit(None).cast(FloatType()))
df_programmi_no_pred = df_programmi_no_pred.withColumn("share_predetto",F.lit(None).cast(DoubleType()))

In [0]:
# Combine all programs and add manual share column
out = inference_spark.unionByName(df_programmi_no_pred, allowMissingColumns=True)
out = out.withColumn('share_manuale', F.lit(None).cast(DoubleType()))
out = out.withColumn("data_str", F.col("Data").cast("string"))

In [0]:
# if table doesn't exist in UC --> create it
if not spark.catalog.tableExists(OUTPUT_TABLE):
    out.write.format("delta").saveAsTable(OUTPUT_TABLE)
else:
    # otherwise do an upsert
    delta_table_comp = DeltaTable.forName(spark, OUTPUT_TABLE)

    (
        delta_table_comp.alias("target")
        .merge(
            out.alias("source"),
            "target.Data = source.Data AND target.Canale = source.Canale AND target.orario_inizio = source.orario_inizio"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

## Predizioni in prime time

In [0]:
# Select columns to show in table for primetime
cols =['Canale', 'Data', 'Programma','programma_norm', 'ORA_INIZIO_TRX', 'ORA_FINE_TRX', 'orario_inizio','orario_fine', 'StoricoShare','share_residuo','share_predetto','share_manuale','Canale_competitor','Programma_competitor','Programma_prev','Programma_next']

In [0]:
# Create primetime only predictions table with a subset of columns and a filter on the time
out_primetime = (
    out
    .select(cols)
    .where(F.col('ORA_INIZIO_TRX') >= START_PRIMETIME) 
)
display(out_primetime)

In [0]:
out_primetime = out_primetime.withColumn(
    'ID',
    F.concat(
        F.col('Canale'),
        F.lit('_'),
        F.col('Data'),
        F.lit('_'),
        F.col('programma_norm'),
        F.lit('_'),
        F.col('orario_inizio')
    )
)

# if table doesn't exist in UC --> create it
if not spark.catalog.tableExists(OUTPUT_TABLE_PRIMETIME):
    out_primetime.write.format("delta").saveAsTable(OUTPUT_TABLE_PRIMETIME)
else:
    # otherwise do an upsert
    delta_table_comp = DeltaTable.forName(spark, OUTPUT_TABLE_PRIMETIME)

    (
        delta_table_comp.alias("target")
        .merge(
            out_primetime.alias("source"),
            "target.Data = source.Data AND target.Canale = source.Canale AND target.orario_inizio = source.orario_inizio"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )